# From overlapping chunks to one entry

`load_corpus` in `src/pipeline.py` joins chunk text with newlines and keeps the overlap. That join is what TF-IDF wants. Mention character offsets cannot use it: the same sentence would be counted twice, and a name split across the overlap would land at two positions.

`deoverlap` keeps the first chunk and appends only the new tail of each later chunk. `load_entries` then collapses whitespace, because the file wraps lines in the middle of a name (`Mrs.` then a newline then `Harker`). Offsets in `data/kg/mentions.jsonl` refer to that cleaned string.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "kg").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(ROOT)


In [ ]:
import json
import re

from src.kg.entries import deoverlap, load_entries, naive_join

raw = json.loads((ROOT / "data" / "dracula_ascii_rag.json").read_text(encoding="utf-8"))
rows = raw["entries"] if isinstance(raw, dict) else raw
chunks = [
    row["text"]
    for row in sorted(rows, key=lambda row: int(row.get("chunk_index") or 0))
    if int(row["entry_index"]) == 1
]
joined = naive_join(chunks)
stripped = deoverlap(chunks)
cleaned = re.sub(r"\s+", " ", stripped).strip()
print("chunks", len(chunks))
print("naive join chars", len(joined))
print("deoverlap chars", len(stripped))
print("cleaned chars", len(cleaned))
print("overlap copies of the opening", joined.count("Left Munich at 8:35"), "vs", cleaned.count("Left Munich at 8:35"))


## The entry record

Each diary entry, letter, or cutting becomes one `Entry`: index, chapter, narrator, dates, a document kind, and the cleaned text. Four entries in this file have no `date_iso`. The narrator strings `Mina Murray` and `Mina Harker` stay distinct here. Merging them is the resolver's job.


In [ ]:
entries = load_entries(ROOT / "data" / "dracula_ascii_rag.json")
first = entries[0]
print(first.entry_index, first.chapter_number, first.narrator, first.date_iso, first.kind)
print(first.text[:180])
print("entries", len(entries))
print("missing date_iso", [e.entry_index for e in entries if e.date_iso is None])
print("Mina narrator labels", sorted({e.narrator for e in entries if e.narrator and e.narrator.startswith("Mina")}))


## Exercise

Print one row per section title: the document kind, then the title. `document_kind` reads the title and the narrator. Journal, diary, letter, newspaper, and phonograph should all appear. `Lucy & Mina (Letters)` and `Newspaper cutting` are document kinds, not people.


In [ ]:
seen = []
for entry in entries:
    key = (entry.kind, entry.section_title)
    if key in seen:
        continue
    seen.append(key)
    print(f"{entry.kind:12} {entry.section_title}")
